<a href="https://colab.research.google.com/github/mas622424/WISER-BQP-QAPINN/blob/main/notebooks/04_QAPINN_5_qubit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install PennyLane for hybrid quantum-classical ML
!pip install -q pennylane
import random
import torch
import torch.nn as nn
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
# Lock the randomness for exact reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Verify GPU availability
device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device_name}")
print(f"PennyLane version: {qml.__version__}")
print(f"PyTorch version: {torch.__version__}")

Using compute device: cuda
PennyLane version: 0.45.1
PyTorch version: 2.11.0+cu128


In [7]:
def compute_burgers_residual(model, x, t, nu=0.01/np.pi):
    """
    Computes the PDE residual for the 1D Viscous Burgers' Equation:
    f = u_t + u * u_x - nu * u_xx = 0
    """
    # Ensure input tensors track gradients for autograd
    x.requires_grad_(True)
    t.requires_grad_(True)

    # Forward pass: predict fluid velocity u(x, t)
    u = model(torch.cat([x, t], dim=1))

    # First-order derivatives: du/dx and du/dt
    u_x = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        retain_graph=True,
        create_graph=True
    )[0]

    u_t = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        retain_graph=True,
        create_graph=True
    )[0]

    # Second-order spatial derivative: d^2u/dx^2
    u_xx = torch.autograd.grad(
        u_x, x,
        grad_outputs=torch.ones_like(u_x),
        retain_graph=True,
        create_graph=True
    )[0]

    # Residual equation (equals 0 when physics is satisfied)
    f_residual = u_t + u * u_x - nu * u_xx
    return f_residual

In [8]:
n_qubits = 5
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # AngleEmbedding automatically handles 2D batched tensors (batch_size, n_qubits)
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")

    # Trainable Ansatz: parameterized rotations + CNOT entanglement
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])  # Ring connection

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}

In [9]:
class QAPINN(nn.Module):
    def __init__(self, n_qubits=4, q_weights_shape=weight_shapes):
        super(QAPINN, self).__init__()

        # Input layer: maps (x, t) coordinates to n_qubits dimensions
        self.input_layer = nn.Linear(2, n_qubits)

        # Quantum Hidden Layer (replaces first classical hidden layer)
        self.quantum_layer = qml.qnn.TorchLayer(quantum_circuit, q_weights_shape)

        # Classical Hidden Layers (using Tanh activation for smooth second derivatives)
        self.hidden1 = nn.Linear(n_qubits, 20)
        self.act1 = nn.Tanh()
        self.hidden2 = nn.Linear(20, 20)
        self.act2 = nn.Tanh()

        # Output layer: predicts scalar fluid velocity u(x, t)
        self.output_layer = nn.Linear(20, 1)

    def forward(self, x_t):
        out = self.input_layer(x_t)
        out = torch.pi * torch.tanh(out)  # Scale to [-pi, pi] for angle embedding
        out = self.quantum_layer(out)
        out = self.act1(self.hidden1(out))
        out = self.act2(self.hidden2(out))
        u = self.output_layer(out)
        return u

In [10]:
def train_qapinn(model, epochs=1000, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()

    # Track trainable parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters in QAPINN: {total_params}\n" + "-"*40)

    loss_history = []

    for epoch in range(epochs):
        optimizer.zero_grad()

        # 1. Collocation points inside the domain (for PDE residual loss)
        x_col = torch.rand(100, 1) * 2 - 1  # x in [-1, 1]
        t_col = torch.rand(100, 1)          # t in [0, 1]
        f_res = compute_burgers_residual(model, x_col, t_col)
        loss_pde = torch.mean(f_res ** 2)

        # 2. Initial Condition: u(x, 0) = -sin(pi * x)
        x_ic = torch.rand(50, 1) * 2 - 1
        t_ic = torch.zeros(50, 1)
        u_ic_pred = model(torch.cat([x_ic, t_ic], dim=1))
        u_ic_true = -torch.sin(np.pi * x_ic)
        loss_ic = mse_loss(u_ic_pred, u_ic_true)

        # 3. Boundary Condition: u(-1, t) = u(1, t) = 0
        t_bc = torch.rand(50, 1)
        x_bc_left = -torch.ones(50, 1)
        x_bc_right = torch.ones(50, 1)
        u_bc_left = model(torch.cat([x_bc_left, t_bc], dim=1))
        u_bc_right = model(torch.cat([x_bc_right, t_bc], dim=1))
        loss_bc = torch.mean(u_bc_left ** 2) + torch.mean(u_bc_right ** 2)

        # Total Physics-Informed Loss
        total_loss = loss_pde + loss_ic + loss_bc
        total_loss.backward()
        optimizer.step()

        loss_history.append(total_loss.item())

        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d} | Total Loss: {total_loss.item():.5f} | PDE Loss: {loss_pde.item():.5f}")

    return loss_history

# Initialize model and run training
qapinn_model = QAPINN(n_qubits=5)
loss_history = train_qapinn(qapinn_model, epochs=1000, lr=0.01)

Total Trainable Parameters in QAPINN: 606
----------------------------------------
Epoch    0 | Total Loss: 0.70913 | PDE Loss: 0.00094
Epoch  100 | Total Loss: 0.12517 | PDE Loss: 0.04826
Epoch  200 | Total Loss: 0.12610 | PDE Loss: 0.04439
Epoch  300 | Total Loss: 0.11864 | PDE Loss: 0.05858
Epoch  400 | Total Loss: 0.09684 | PDE Loss: 0.05279
Epoch  500 | Total Loss: 0.12106 | PDE Loss: 0.04455
Epoch  600 | Total Loss: 0.11477 | PDE Loss: 0.05612
Epoch  700 | Total Loss: 0.11941 | PDE Loss: 0.05771
Epoch  800 | Total Loss: 0.11256 | PDE Loss: 0.04328
Epoch  900 | Total Loss: 0.12208 | PDE Loss: 0.04132


In [11]:
import numpy as np

print("Generating evaluation grid and extracting artifacts for Role 3...")

# 1. Create a 100x100 grid for (x, t)
x_grid = torch.linspace(-1, 1, 256)
t_grid = torch.linspace(0, 1, 100)
X, T = torch.meshgrid(x_grid, t_grid, indexing="ij")
inputs = torch.cat([X.reshape(-1, 1), T.reshape(-1, 1)], dim=1)

# 2. Setup PyTorch Hooks to capture internal Layer 2 and Layer 3 activations
activations = {}
def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach().numpy()
    return hook

# Attach hooks to the Tanh activations of your hidden layers
h1 = qapinn_model.act1.register_forward_hook(get_activation('layer2'))
h2 = qapinn_model.act2.register_forward_hook(get_activation('layer3'))

# 3. Run the grid through the trained model
with torch.no_grad():
    u_pred_flat = qapinn_model(inputs).numpy()

# Remove hooks so they don't slow down future training
h1.remove()
h2.remove()

# 4. Format and Save Predictions (shape: 100x100)
u_pred = u_pred_flat.reshape(256, 100)
np.save('predictions_qapinn_5q.npy', u_pred)

# 5. Format and Save Activations (shape: 20 neurons, 100x100 grid)
# We transpose so the shape is exactly what Role 3 requested: (n_neurons, nx, nt)
act_layer2 = activations['layer2'].reshape(256, 100, 20).transpose(2, 0, 1)
act_layer3 = activations['layer3'].reshape(256, 100, 20).transpose(2, 0, 1)
np.savez('activations_qapinn_5q.npz', layer2=act_layer2, layer3=act_layer3)

# 6. Save Loss Curve
np.save('loss_curve_qapinn_5q.npy', np.array(loss_history))

# 7. Save Flattened Weights (for Loss Landscape analysis)
weights_list = [p.detach().numpy().flatten() for p in qapinn_model.parameters()]
flat_weights = np.concatenate(weights_list)
np.savez('weights_qapinn_5q.npz', weights=flat_weights)

# 8. Print Summary for Role 3
total_params = sum(p.numel() for p in qapinn_model.parameters() if p.requires_grad)
print("\n EXPORT COMPLETE!")
print(f"Total Trainable Parameters: {total_params}")
print("Files generated in Colab filesystem:")
print(" - predictions_qapinn_5q.npy")
print(" - activations_qapinn_5q.npz")
print(" - loss_curve_qapinn_5q.npy")
print(" - weights_qapinn_5q.npz")

Generating evaluation grid and extracting artifacts for Role 3...

 EXPORT COMPLETE!
Total Trainable Parameters: 606
Files generated in Colab filesystem:
 - predictions_qapinn_5q.npy
 - activations_qapinn_5q.npz
 - loss_curve_qapinn_5q.npy
 - weights_qapinn_5q.npz


In [12]:
import scipy.io
import urllib.request
import numpy as np

print("Downloading exact analytical solution for Burgers' Equation...")
# Fetch the standard dataset from the original PINNs repository
url = "https://raw.githubusercontent.com/maziarraissi/PINNs/master/appendix/Data/burgers_shock.mat"
urllib.request.urlretrieve(url, "burgers_shock.mat")

# Load the MATLAB file
data = scipy.io.loadmat('burgers_shock.mat')

# 'usol' is the exact fluid velocity u(x,t)
Exact = np.real(data['usol']).T

# Save it as ground_truth.npy for Role 3!
np.save('ground_truth.npy', Exact)

print(" ground_truth.npy saved successfully! Send this to Role 3.")

 ground_truth.npy saved successfully! Send this to Role 3.


In [13]:
print("Calculating PDE Residual for Role 3...")

# 1. Prepare grid with gradients enabled
x_flat = X.reshape(-1, 1).clone().detach().requires_grad_(True)
t_flat = T.reshape(-1, 1).clone().detach().requires_grad_(True)

# 2. Compute residual using your existing function
f_res = compute_burgers_residual(qapinn_model, x_flat, t_flat)

# 3. Reshape to match the 256x100 grid and save
pde_residual_grid = f_res.detach().numpy().reshape(256, 100)
np.save('pde_residual_qapinn_5q.npy', pde_residual_grid)

print("Done! pde_residual_qapinn_5q.npy has been saved.")

Calculating PDE Residual for Role 3...
Done! pde_residual_qapinn_5q.npy has been saved.
